# 04 · Is the EEGNet AUC ≈ 0.61 real or a fluke?

The diagnostics notebook gave **EEGNet (raw) AUC 0.61** on one split with one seed, while CSP+SVM and the first EEGNet run (0.51) sat at chance. This notebook settles it with three arms, all using the **plain label mapping** (file k ↔ label row k) and the **same native EEGNet as notebook 03**:

| Arm | What | Answers |
|---|---|---|
| **A** | EDA split (train/val/test blocks), **5 seeds** | does 0.61 survive re-initialisation? |
| **B** | **Rotating test sessions**: 5-fold CV over blocks of 45 files (= 5 runs), 5 seeds, every trial is tested exactly once per seed, with an embargo around test/val blocks | does it generalise to *other* sessions, not just one lucky split? |
| **C** | Same as B but with **labels shuffled within each block** (class balance kept) | what AUC does this exact procedure produce when there is *nothing* to learn? |

**Decision rule (fixed before running).** The signal is called **CONFIRMED** only if *all* hold for arm B:
1. the 95% block-bootstrap interval of the 5-seed ensemble AUC lies **above 0.5**;
2. mean real AUC exceeds the **largest** AUC of the arm-C control runs;
3. at least **4 of 5** test folds have mean AUC > 0.5.

Otherwise it is **NOT CONFIRMED**: treat 0.61 as a fluke and the models as chance-level on this dataset. (A true AUC of ~0.52 would not be detectable here; with ~1900 trials, the AUC standard error is ≈0.013.)

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
warnings.filterwarnings("ignore")
torch.set_num_threads(max(1, os.cpu_count() or 1))

_cwd = Path.cwd().resolve()
ROOT = Path(os.environ["BCI_ROOT"]) if os.environ.get("BCI_ROOT") else (_cwd.parent if _cwd.name.lower() == "notebooks" else _cwd)
PREP = ROOT / "bci_prepared"; PRE, SPL = PREP / "02_preprocessed", PREP / "05_splits"
REP = PREP / "06_reports" / "diagnostics"; REP.mkdir(parents=True, exist_ok=True)

N_SEEDS  = int(os.environ.get("STAB_SEEDS", 5))
N_NULL   = int(os.environ.get("STAB_NULL", 5))
EPOCHS   = int(os.environ.get("STAB_EPOCHS", 60))
PATIENCE = int(os.environ.get("STAB_PATIENCE", 12))
K_FOLDS, BLOCK, EMBARGO, VAL_FRAC = 5, int(os.environ.get("STAB_BLOCK", 45)), 11, 0.15
N_BOOT = 1000

X = np.load(PRE / "X_preprocessed.npy").astype(np.float32); y = np.load(PRE / "y.npy").astype(int); ids = np.load(PRE / "file_ids.npy")
assert X.shape[0] == len(y) == len(ids) and X.shape[1] == 4 and np.all(np.diff(ids) > 0)
pos = ids - ids.min(); blk = pos // BLOCK
print("X", X.shape, "| P(Right) =", y.mean().round(3), "| blocks of", BLOCK, "files:", len(np.unique(blk)), "| seeds", N_SEEDS, "| null runs", N_NULL, "| epochs<=", EPOCHS)

In [ ]:
class EEGNet(nn.Module):
    # identical to notebook 03 (EEGNet-8,2 for 4 channels)
    def __init__(s, C, T, F1=8, D=2, F2=16, k=125, p=0.5):
        super().__init__()
        s.b1 = nn.Sequential(nn.Conv2d(1, F1, (1, k), padding=(0, k // 2), bias=False), nn.BatchNorm2d(F1), nn.Conv2d(F1, F1 * D, (C, 1), groups=F1, bias=False), nn.BatchNorm2d(F1 * D), nn.ELU(), nn.AvgPool2d((1, 4)), nn.Dropout(p))
        s.b2 = nn.Sequential(nn.Conv2d(F1 * D, F1 * D, (1, 16), padding=(0, 8), groups=F1 * D, bias=False), nn.Conv2d(F1 * D, F2, 1, bias=False), nn.BatchNorm2d(F2), nn.ELU(), nn.AvgPool2d((1, 8)), nn.Dropout(p))
        with torch.no_grad(): n = s.b2(s.b1(torch.zeros(1, 1, C, T))).numel()
        s.fc = nn.Linear(n, 2)
    def forward(s, x): return s.fc(s.b2(s.b1(x.unsqueeze(1))).flatten(1))

def train_predict(Xtr, ytr, Xva, yva, Xte, seed, epochs=EPOCHS, patience=PATIENCE):
    # trains on (tr), early-stops on (va), returns P(Right) for (te)
    torch.manual_seed(seed); np.random.seed(seed)
    sc = np.median(Xtr.std(axis=2), axis=0)[None, :, None] + 1e-8
    T = lambda a: torch.from_numpy(np.clip(np.nan_to_num(a / sc, nan=0.0, posinf=0.0, neginf=0.0), -15, 15).astype(np.float32))
    m = EEGNet(4, Xtr.shape[2]); opt = torch.optim.AdamW(m.parameters(), 1e-3, weight_decay=1e-2); lf = nn.CrossEntropyLoss()
    Ttr, Tva, Tte = T(Xtr), T(Xva), T(Xte)
    dl = DataLoader(TensorDataset(Ttr, torch.from_numpy(ytr).long()), batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(seed))
    def pr(a):
        m.eval(); out = []
        with torch.no_grad():
            for i in range(0, len(a), 256): out.append(torch.softmax(m(a[i:i + 256]), 1)[:, 1].numpy())
        return np.nan_to_num(np.concatenate(out), nan=0.5)
    best, wait = -9, 0; state = {k: v.clone() for k, v in m.state_dict().items()}
    for ep in range(epochs):
        m.train()
        for xb, yb in dl: opt.zero_grad(); lf(m(xb + 0.05 * torch.randn_like(xb)), yb).backward(); opt.step()
        pv = pr(Tva)
        try: b = balanced_accuracy_score(yva, pv > .5) + 0.001 * roc_auc_score(yva, pv)
        except ValueError: b = balanced_accuracy_score(yva, pv > .5)
        if b > best: best, wait, state = b, 0, {k: v.clone() for k, v in m.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break
    m.load_state_dict(state); return pr(Tte), ep + 1

def block_boot_auc(yv, pv, groups, n=N_BOOT, seed=0):
    r = np.random.default_rng(seed); ug = np.unique(groups); idx = {g: np.where(groups == g)[0] for g in ug}; out = []
    for _ in range(n):
        ii = np.concatenate([idx[g] for g in r.choice(ug, len(ug))])
        if len(np.unique(yv[ii])) == 2: out.append(roc_auc_score(yv[ii], pv[ii]))
    return np.percentile(out, [2.5, 97.5])

## Arm A — same EDA split, 5 seeds
Uses `05_splits/split_assignment.csv` (all non-embargo trials). Differs from notebook 03 only in that the test set is **not** reduced to the sessions ≥ 10 files, so it is the full test split (the first EEGNet run in notebook 02 used this same split and scored AUC 0.51).

In [ ]:
armA = None; sp_file = SPL / "split_assignment.csv"
if sp_file.exists():
    sp = pd.read_csv(sp_file).set_index("file_id"); spl = pd.Series(ids).map(sp["split"]).fillna("EMBARGO").to_numpy()
    tr, va, te = [np.where(spl == s)[0] for s in ("train", "val", "test")]; print("train/val/test:", len(tr), len(va), len(te))
    rows, P = [], []; t0 = time.time()
    for s in range(N_SEEDS):
        p, ep = train_predict(X[tr], y[tr], X[va], y[va], X[te], seed=s); P.append(p)
        rows.append(dict(seed=s, auc=roc_auc_score(y[te], p), bal_acc=balanced_accuracy_score(y[te], p > .5), epochs=ep)); print(rows[-1], f"[{time.time()-t0:.0f}s]")
    A = pd.DataFrame(rows); pe = np.mean(P, 0); lo, hi = block_boot_auc(y[te], pe, blk[te])
    armA = dict(per_seed=rows, mean_auc=float(A.auc.mean()), sd_auc=float(A.auc.std()), ensemble_auc=float(roc_auc_score(y[te], pe)), ensemble_ci=[float(lo), float(hi)], n_test=int(len(te)))
    display(A.round(4)); print(f"Arm A: mean AUC {A.auc.mean():.3f} ± {A.auc.std():.3f} | ensemble AUC {armA['ensemble_auc']:.3f} (95% block-bootstrap CI {lo:.3f}–{hi:.3f}) | n_test={len(te)}")
else: print("no split_assignment.csv -> arm A skipped")

## Arms B and C — rotating test sessions, real vs shuffled labels
For each seed the 45-file blocks are randomly assigned to 5 folds. Per fold: test = that fold's blocks; validation = 15 % of the remaining blocks (early stopping only); training = the rest **minus any trial within 11 file positions of a test or validation trial** (the EDA's decorrelation lag).
Arm C repeats everything with labels permuted **within each block**.

In [ ]:
def dist_to_set(p, held):
    held = np.sort(held); i = np.searchsorted(held, p); lo = np.abs(p - held[np.clip(i - 1, 0, len(held) - 1)]); hi = np.abs(p - held[np.clip(i, 0, len(held) - 1)]); return np.minimum(lo, hi)

def rotating_cv(y_used, seed):
    r = np.random.default_rng(1000 + seed); ub = np.unique(blk); order = r.permutation(len(ub)); fob = {ub[b]: i % K_FOLDS for i, b in enumerate(order)}
    fold = np.array([fob[b] for b in blk]); oof = np.full(len(y_used), np.nan); fold_auc = []
    for f in range(K_FOLDS):
        te = np.where(fold == f)[0]; rest = np.setdiff1d(np.arange(len(y_used)), te)
        rb = np.unique(blk[rest]); vb = r.choice(rb, max(1, int(VAL_FRAC * len(rb))), replace=False)
        va = rest[np.isin(blk[rest], vb)]; trc = np.setdiff1d(rest, va)
        trc = trc[dist_to_set(pos[trc], pos[np.concatenate([te, va])]) > EMBARGO]
        p, _ = train_predict(X[trc], y_used[trc], X[va], y_used[va], X[te], seed=seed); oof[te] = p
        fold_auc.append(roc_auc_score(y_used[te], p) if len(np.unique(y_used[te])) == 2 else np.nan)
    return oof, np.array(fold_auc)

def shuffle_within_blocks(yv, seed):
    r = np.random.default_rng(5000 + seed); yp = yv.copy()
    for b in np.unique(blk): i = np.where(blk == b)[0]; yp[i] = r.permutation(yv[i])
    return yp

resB, resC, oofB, foldB = [], [], [], []; t0 = time.time()
for s in range(N_SEEDS):
    oof, fa = rotating_cv(y, s); oofB.append(oof); foldB.append(fa)
    resB.append(dict(seed=s, auc=roc_auc_score(y, oof), bal_acc=balanced_accuracy_score(y, oof > .5))); print("B", resB[-1], "fold AUCs", fa.round(3), f"[{time.time()-t0:.0f}s]")
for s in range(N_NULL):
    yp = shuffle_within_blocks(y, s); oof, _ = rotating_cv(yp, s)
    resC.append(dict(seed=s, auc=roc_auc_score(yp, oof), bal_acc=balanced_accuracy_score(yp, oof > .5))); print("C", resC[-1], f"[{time.time()-t0:.0f}s]")
B, Cn = pd.DataFrame(resB), pd.DataFrame(resC)

## Verdict

In [ ]:
pe = np.mean(oofB, 0); ens_auc = roc_auc_score(y, pe); lo, hi = block_boot_auc(y, pe, blk)
fold_mean = np.nanmean(np.vstack(foldB), axis=0); n_fold_up = int((fold_mean > 0.5).sum())
c1, c2, c3 = lo > 0.5, B.auc.mean() > Cn.auc.max(), n_fold_up >= 4
print(f"Arm B (real labels):     per-seed AUC {B.auc.round(3).tolist()} | mean {B.auc.mean():.3f} ± {B.auc.std():.3f} | bal-acc mean {B.bal_acc.mean():.3f}")
print(f"Arm C (shuffled labels): per-seed AUC {Cn.auc.round(3).tolist()} | mean {Cn.auc.mean():.3f} ± {Cn.auc.std():.3f} | max {Cn.auc.max():.3f}")
print(f"Ensemble AUC (mean of {N_SEEDS} seeds) {ens_auc:.3f} | 95% block-bootstrap CI {lo:.3f}–{hi:.3f}")
print(f"Mean AUC per test fold (over seeds): {fold_mean.round(3).tolist()}  -> {n_fold_up}/{K_FOLDS} above 0.5")
print(f"\nCriteria: (1) CI lower bound > 0.5: {c1} | (2) mean real AUC > max shuffled AUC: {c2} | (3) >=4/5 folds above 0.5: {c3}")
verdict = "CONFIRMED: a weak but reproducible signal exists with the plain mapping" if (c1 and c2 and c3) else "NOT CONFIRMED: the earlier 0.61 is consistent with chance; EEGNet does not beat chance when tested on other sessions"
print("VERDICT:", verdict)

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.scatter(np.zeros(len(B)) + np.random.default_rng(0).uniform(-.08, .08, len(B)), B.auc, c="tab:red", label="B real labels (per seed)")
ax.scatter(np.ones(len(Cn)) + np.random.default_rng(1).uniform(-.08, .08, len(Cn)), Cn.auc, c="gray", label="C shuffled labels")
if armA: ax.scatter(np.full(len(armA["per_seed"]), -1.0) + np.random.default_rng(2).uniform(-.08, .08, len(armA["per_seed"])), [r["auc"] for r in armA["per_seed"]], c="tab:blue", label="A EDA split (per seed)")
ax.errorbar([0], [ens_auc], yerr=[[ens_auc - lo], [hi - ens_auc]], fmt="k_", capsize=6, label="ensemble AUC, 95% CI"); ax.axhline(.5, c="k", ls=":")
ax.set_xticks([-1, 0, 1]); ax.set_xticklabels(["A: EDA split", "B: rotating CV", "C: shuffled"]); ax.set_ylabel("AUC"); ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(REP / "eegnet_stability.png", dpi=140); plt.show()

(REP / "eegnet_stability.json").write_text(json.dumps(dict(armA=armA, armB=dict(per_seed=B.to_dict("records"), ensemble_auc=float(ens_auc), ensemble_ci=[float(lo), float(hi)], fold_mean_auc=fold_mean.tolist()),
    armC=dict(per_seed=Cn.to_dict("records")), criteria=dict(ci_above_half=bool(c1), beats_shuffled=bool(c2), folds_above_half=int(n_fold_up)), verdict=verdict,
    settings=dict(seeds=N_SEEDS, null_runs=N_NULL, epochs=EPOCHS, patience=PATIENCE, block=BLOCK, embargo=EMBARGO, folds=K_FOLDS)), indent=2, default=float))
np.save(REP / "eegnet_oof_ensemble.npy", pe); print("saved ->", REP)

**Send back:** this executed notebook and `bci_prepared/06_reports/diagnostics/eegnet_stability.json`. Runtime is roughly 15–40 min on CPU; `STAB_SEEDS`, `STAB_NULL`, `STAB_EPOCHS` shorten it.